# FMUCD Preprocessing (Leakage-Safe)

This notebook loads the FMUCD dataset, cleans it, engineers features, computes time-aware historical UPM rates, and creates leakage-safe train/val splits with categorical encoding.


## 1. Setup


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


## 2. Configuration
Adjust paths or parameters here if needed.


In [2]:
# Paths
INPUT_PATH = Path("/Users/mehakxoxo/Documents/spring_2026/data_practicum/Facility Management Unified Classification Database (FMUCD)/Facility Management Unified Classification Database (FMUCD).csv")
OUTPUT_DIR = Path("/Users/mehakxoxo/Documents/spring_2026/data_practicum/ai-predictive-maintenance-capstone/data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns
DATE_COLS = ["WOStartDate", "WOEndDate"]
CATEGORICAL_COLS = [
    "SystemDescription",
    "SubsystemDescription",
    "ComponentDescription",
    "WOPriority",
    "Type",
    "Country",
    "State/Province",
]

WEATHER_RENAME = {
    "MinTemp.(°C)": "MinTemp",
    "MaxTemp.(°C)": "MaxTemp",
    "Atmospheric pressure(hPa)": "Pressure",
    "Humidity(%)": "Humidity",
    "WindSpeed(m/s)": "WindSpeed",
    "Precipitation(mm)": "Precipitation",
    "Snow(mm)": "Snow",
}

WEATHER_COLS = [
    "MinTemp",
    "MaxTemp",
    "Humidity",
    "WindSpeed",
    "Precipitation",
    "Snow",
    "Pressure",
]

LABEL_CANDIDATES = [
    "PPM/UPM",
    "PPM_UPM",
    "MaintenanceType",
    "WorkOrderType",
    "Label",
]

# Feature engineering params (Celsius)
FREEZE_THRESH = 0.0
EXTREME_HEAT_THRESH = 35.0
EXTREME_COLD_THRESH = -10.0

# Encoding
ENCODING = "ordinal"  # "ordinal" or "onehot"
MAX_CATEGORIES = 50
MIN_FREQUENCY = 0.01

# Split
VAL_PCT = 0.2


## 3. Helper functions


In [3]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns=WEATHER_RENAME)
    return df


def infer_label_col(df: pd.DataFrame, explicit: str = "") -> str:
    if explicit and explicit in df.columns:
        return explicit
    for col in LABEL_CANDIDATES:
        if col in df.columns:
            return col
    for col in df.columns:
        if "ppm" in col.lower() and "upm" in col.lower():
            return col
    return ""


def coerce_dates(df: pd.DataFrame, date_cols) -> pd.DataFrame:
    df = df.copy()
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def add_calendar_features(df: pd.DataFrame, start_col: str) -> pd.DataFrame:
    df = df.copy()
    if start_col not in df.columns:
        return df
    start = df[start_col]
    df["Month"] = start.dt.month
    df["DayOfWeek"] = start.dt.dayofweek
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

    df["Season"] = pd.cut(
        df["Month"],
        bins=[0, 2, 5, 8, 11, 12],
        labels=["Winter", "Spring", "Summer", "Fall", "Winter"],
        include_lowest=True,
        ordered=False,
    ).astype(str)

    return df


def add_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "MinTemp" in df.columns:
        df["FreezeFlag"] = (df["MinTemp"] <= FREEZE_THRESH).astype(int)
    else:
        df["FreezeFlag"] = 0

    if "Snow" in df.columns:
        df["SnowFlag"] = (df["Snow"] > 0).astype(int)
    else:
        df["SnowFlag"] = 0

    if "Precipitation" in df.columns:
        df["RainFlag"] = (df["Precipitation"] > 0).astype(int)
    else:
        df["RainFlag"] = 0

    if "MinTemp" in df.columns and "MaxTemp" in df.columns:
        df["TempRange"] = df["MaxTemp"] - df["MinTemp"]
        df["ExtremeHeatFlag"] = (df["MaxTemp"] >= EXTREME_HEAT_THRESH).astype(int)
        df["ExtremeColdFlag"] = (df["MinTemp"] <= EXTREME_COLD_THRESH).astype(int)
    else:
        df["TempRange"] = np.nan
        df["ExtremeHeatFlag"] = 0
        df["ExtremeColdFlag"] = 0

    return df


def fill_missing(df: pd.DataFrame, categorical_cols) -> pd.DataFrame:
    df = df.copy()
    categorical_cols = [c for c in categorical_cols if c in df.columns]

    for col in categorical_cols:
        df[col] = df[col].astype("string").str.strip().fillna("Unknown")

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        median = df[col].median()
        if pd.isna(median):
            continue
        df[col] = df[col].fillna(median)

    return df


def map_upm_label(series: pd.Series) -> pd.Series:
    if series.dtype.kind in {"i", "u", "f"}:
        return series.astype(float)

    s = series.astype("string").str.strip().str.upper()
    return s.map({"UPM": 1.0, "PPM": 0.0, "1": 1.0, "0": 0.0})


def add_historical_upm_rates(df: pd.DataFrame, label_col: str) -> pd.DataFrame:
    if not label_col or label_col not in df.columns:
        return df

    df = df.copy()
    df["UPMLabel"] = map_upm_label(df[label_col])

    if "WOStartDate" in df.columns:
        df = df.sort_values("WOStartDate")

    df["UPMRate_Overall"] = df["UPMLabel"].expanding().mean().shift(1)

    if "SystemDescription" in df.columns:
        df["UPMRate_System"] = (
            df.groupby("SystemDescription")["UPMLabel"]
            .expanding()
            .mean()
            .shift(1)
            .reset_index(level=0, drop=True)
        )

    if "SystemDescription" in df.columns and "SubsystemDescription" in df.columns:
        df["UPMRate_SystemSubsystem"] = (
            df.groupby(["SystemDescription", "SubsystemDescription"])["UPMLabel"]
            .expanding()
            .mean()
            .shift(1)
            .reset_index(level=[0, 1], drop=True)
        )

    global_mean = df["UPMLabel"].mean()
    for col in ["UPMRate_System", "UPMRate_SystemSubsystem", "UPMRate_Overall"]:
        if col in df.columns:
            df[col] = df[col].fillna(df["UPMRate_Overall"]).fillna(global_mean)

    return df


def compute_duration(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "WODuration" in df.columns and df["WODuration"].notna().any():
        return df
    if "WOStartDate" in df.columns and "WOEndDate" in df.columns:
        df["WODuration"] = (df["WOEndDate"] - df["WOStartDate"]).dt.total_seconds() / 3600.0
    return df


def time_split(df: pd.DataFrame, date_col: str, val_pct: float):
    if date_col in df.columns:
        df = df.sort_values(date_col)
    else:
        df = df.sample(frac=1.0, random_state=42)

    split_idx = int(len(df) * (1 - val_pct))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def reduce_categories(train: pd.DataFrame, val: pd.DataFrame, categorical_cols, max_categories, min_frequency):
    cat_cols = [c for c in categorical_cols if c in train.columns]
    if not cat_cols:
        return train, val

    train_reduced = train.copy()
    val_reduced = val.copy()

    for col in cat_cols:
        vc = train_reduced[col].astype("string").value_counts(normalize=True, dropna=False)
        keep = vc[vc >= min_frequency].head(max_categories).index
        train_reduced[col] = train_reduced[col].where(train_reduced[col].isin(keep), "Other")
        val_reduced[col] = val_reduced[col].where(val_reduced[col].isin(keep), "Other")

    return train_reduced, val_reduced


def encode_onehot(train: pd.DataFrame, val: pd.DataFrame, categorical_cols):
    cat_cols = [c for c in categorical_cols if c in train.columns]
    if not cat_cols:
        return train, val

    train_enc = pd.get_dummies(train, columns=cat_cols, dummy_na=False)
    val_enc = pd.get_dummies(val, columns=cat_cols, dummy_na=False)
    val_enc = val_enc.reindex(columns=train_enc.columns, fill_value=0)
    return train_enc, val_enc


def encode_ordinal(train: pd.DataFrame, val: pd.DataFrame, categorical_cols):
    cat_cols = [c for c in categorical_cols if c in train.columns]
    if not cat_cols:
        return train, val

    train_enc = train.copy()
    val_enc = val.copy()
    for col in cat_cols:
        categories = pd.Series(train_enc[col].astype("string")).dropna().unique().tolist()
        train_enc[col] = pd.Categorical(train_enc[col], categories=categories).codes
        val_enc[col] = pd.Categorical(val_enc[col], categories=categories).codes
    return train_enc, val_enc


## 4. Load data


In [4]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
df = normalize_columns(df)

label_col = infer_label_col(df)
label_col


'PPM/UPM'

## 5. Preprocess + Feature Engineering


In [5]:
df = coerce_dates(df, DATE_COLS)
df = compute_duration(df)
df = add_calendar_features(df, "WOStartDate")
df = add_weather_features(df)
df = add_historical_upm_rates(df, label_col)
df = fill_missing(df, CATEGORICAL_COLS + ["Season"])

# Time-based split
train, val = time_split(df, "WOStartDate", VAL_PCT)

# Reduce category cardinality before one-hot
if ENCODING == "onehot":
    train, val = reduce_categories(train, val, CATEGORICAL_COLS + ["Season"], MAX_CATEGORIES, MIN_FREQUENCY)
    train_enc, val_enc = encode_onehot(train, val, CATEGORICAL_COLS + ["Season"])
else:
    train_enc, val_enc = encode_ordinal(train, val, CATEGORICAL_COLS + ["Season"])

train_enc.head()


,UniversityID,Country,State/Province,BuildingID,BuildingName,Size,Type,BuiltYear,FCI (facility condition index),CRV (current replacement value),...,FreezeFlag,SnowFlag,RainFlag,TempRange,ExtremeHeatFlag,ExtremeColdFlag,UPMLabel,UPMRate_Overall,UPMRate_System,UPMRate_SystemSubsystem
1118757,5,0,0,0055,NaN,121585.0,0,1969.0,0.426493,49627722.94,...,0,0,0,2.310417,0,0,1.0,0.397741,0.352849,0.242214
1118756,5,0,0,0081,NaN,121585.0,0,1969.0,0.426493,49627722.94,...,0,0,0,1.009583,0,0,1.0,1.000000,0.245969,0.440939
1118754,5,0,0,0003,NaN,121585.0,0,1969.0,0.426493,49627722.94,...,1,0,0,2.332500,0,0,0.0,1.000000,0.750000,0.311464
1118755,5,0,0,0003,NaN,121585.0,0,1969.0,0.426493,49627722.94,...,1,0,0,2.332500,0,0,0.0,0.666667,0.000000,0.000000
1118752,5,0,0,0030,NaN,121585.0,0,1969.0,0.426493,49627722.94,...,1,0,0,2.332500,0,0,0.0,0.500000,0.000000,0.403763


## 6. Save outputs


In [6]:
# Save cleaned full data (large). Comment out if you only want train/val.
# df.to_csv(OUTPUT_DIR / "fmucd_clean.csv", index=False)

# Save train/val (compressed to save space)
train_enc.to_csv(OUTPUT_DIR / "fmucd_clean_train.csv.gz", index=False, compression="gzip")
val_enc.to_csv(OUTPUT_DIR / "fmucd_clean_val.csv.gz", index=False, compression="gzip")

# Simple report
report = {
    "rows_full": int(len(df)),
    "rows_train": int(len(train_enc)),
    "rows_val": int(len(val_enc)),
    "columns_full": int(df.shape[1]),
    "encoding": ENCODING,
    "max_categories": int(MAX_CATEGORIES),
    "min_frequency": float(MIN_FREQUENCY),
    "label_col": label_col,
    "missingness_top20": df.isna().mean().sort_values(ascending=False).head(20).round(4).to_dict(),
}

(OUTPUT_DIR / "fmucd_clean_report.json").write_text(json.dumps(report, indent=2))

report


{'rows_full': 3731442,
 'rows_train': 2985153,
 'rows_val': 746289,
 'columns_full': 52,
 'encoding': 'ordinal',
 'max_categories': 50,
 'min_frequency': 0.01,
 'label_col': 'PPM/UPM',
 'missingness_top20': {'DMC (deferred maintenance cost)': 0.8848,
  'BuildingName': 0.615,
  'WOEndDate': 0.4071,
  'WOStartDate': 0.3652,
  'BuildingID': 0.2249,
  'DescriptiveCode': 0.0155,
  'SubsystemCode': 0.0155,
  'SystemCode': 0.0155,
  'PPM/UPM': 0.0101,
  'WOID': 0.0018,
  'WODescription': 0.0003,
  'Season': 0.0,
  'Month': 0.0,
  'Cloudness(%)': 0.0,
  'IsWeekend': 0.0,
  'Snow': 0.0,
  'Precipitation': 0.0,
  'WindDegree': 0.0,
  'DayOfWeek': 0.0,
  'UniversityID': 0.0}}